In [ ]:
import openai
import psycopg2  # or use sqlite3
import sqlparse
import re
import json
import plotly.io as pio
import os
from dotenv import load_dotenv

load_dotenv(override=True)
# openai.api_key = "your-openai-key"
openai.api_key = os.getenv("OPENAI_API_KEY")

print("Loaded OpenAI Key:", openai.api_key[-8:] + "..." if openai.api_key else "Not found!")

# === 1. Connect to database ===
def get_connection():
    return psycopg2.connect(
        dbname="raccoon",
        user="postgres",
        password="postgres",
        host="localhost",
        port=5433
    )

# === 2. Get schema with sample rows ===
def get_schema_with_samples(conn):
    cursor = conn.cursor()
    schema = ""

    cursor.execute("""SELECT table_name FROM information_schema.tables 
                      WHERE table_schema='public' AND table_type='BASE TABLE';""")
    tables = cursor.fetchall()

    for (table,) in tables:
        cursor.execute(f"SELECT column_name, data_type FROM information_schema.columns WHERE table_name = '{table}'")
        columns = cursor.fetchall()
        schema += f"\nTable: {table}\nColumns:\n"
        schema += "\n".join([f"  - {col} ({dtype})" for col, dtype in columns])

        cursor.execute(f"SELECT * FROM {table} LIMIT 3")
        rows = cursor.fetchall()
        schema += f"\nSample rows:\n{rows}\n"

    return schema.strip()

# === 3. Validate query (must be safe) ===
def is_safe_sql(query):
    parsed = sqlparse.parse(query)
    for statement in parsed:
        tokens = [token.value.lower() for token in statement.tokens if not token.is_whitespace]
        if not any(token.startswith("select") for token in tokens):
            return False
        if any(re.search(r"\b(password|phone|bank|email|delete|drop|update|insert|grant|revoke)\b", token) for token in tokens):
            return False
    return True

# === 4. Run query locally ===
def run_sql(query, conn):
    cursor = conn.cursor()
    cursor.execute(query)
    columns = [desc[0] for desc in cursor.description]
    rows = cursor.fetchall()
    return {"columns": columns, "rows": rows}

# === 5. Ask GPT for query generation ===
def generate_sql(user_question, schema_context):
    messages = [
        {"role": "system", "content": f"You are an assistant that generates SQL queries. Use only SELECT and try to use only meaningful columns, not just ids, use joins only when necessary and do not confuse table name aliases. Schema:\n{schema_context}"},
        {"role": "user", "content": user_question}
    ]
    print(schema_context)
    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content.strip()

# === 6. Generate Plotly JSON ===
def generate_plotly_json(data, user_question):
    messages = [
        {"role": "system", "content": "You generate Plotly chart JSON (not full HTML). Use only plotly.graph_objects or plotly.express, return just the fig.to_json() output."},
        {"role": "user", "content": f"The user asked: '{user_question}'. Here is the data:\nColumns: {data['columns']}\nRows: {data['rows']}"}
    ]
    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content.strip()

# === 7. Full pipeline ===
def handle_user_query(user_question):
    conn = get_connection()
    schema = get_schema_with_samples(conn)

    sql_query = generate_sql(user_question, schema).replace("`","").replace("sql","")
    print("Generated SQL:", sql_query)

    if not is_safe_sql(sql_query):
        raise Exception("Unsafe or prohibited SQL query.")

    data = run_sql(sql_query, conn)
    plotly_json = generate_plotly_json(data, user_question)
    # print("Plotly json:", plotly_json)
    conn.close()
    # return json.loads(plotly_json)
    return plotly_json

# === Example usage ===
# if __name__ == "__main__":
#     user_question = "What are the top 5 drivers by amount of routes?"
#     try:
#         fig_json = handle_user_query(user_question)
#         fig = pio.from_json(json.dumps(fig_json))
#         fig.show()  # Optional, for debugging
#     except Exception as e:
#         print("Error:", e)


Loaded OpenAI Key: JIqru1EA...


In [2]:
user_question = "What are the top 5 drivers by amount of routes?"

In [3]:
fig_json = handle_user_query(user_question)


Generated SQL: 
SELECT driver_id, COUNT(id) AS route_count
FROM driver_on_shift
GROUP BY driver_id
ORDER BY route_count DESC
LIMIT 5;



In [4]:
fig_json

'```json\n{\n    "data": [\n        {\n            "type": "bar",\n            "x": ["Driver 33", "Driver 4", "Driver 34", "Driver 41", "Driver 9"],\n            "y": [24, 20, 19, 18, 17]\n        }\n    ],\n    "layout": {\n        "title": {\n            "text": "Top 5 Drivers by Amount of Routes"\n        },\n        "xaxis": {\n            "title": {\n                "text": "Driver ID"\n            }\n        },\n        "yaxis": {\n            "title": {\n                "text": "Route Count"\n            }\n        }\n    }\n}\n```'

In [5]:
print(fig_json)

```json
{
    "data": [
        {
            "type": "bar",
            "x": ["Driver 33", "Driver 4", "Driver 34", "Driver 41", "Driver 9"],
            "y": [24, 20, 19, 18, 17]
        }
    ],
    "layout": {
        "title": {
            "text": "Top 5 Drivers by Amount of Routes"
        },
        "xaxis": {
            "title": {
                "text": "Driver ID"
            }
        },
        "yaxis": {
            "title": {
                "text": "Route Count"
            }
        }
    }
}
```


In [6]:
fig = pio.from_json(json.dumps(json.loads(fig_json.replace("`","").replace("json",""))))


In [7]:
fig.show()

In [43]:
query = """SELECT "user"."first_name", "user"."last_name", COUNT("job"."id") AS "total_jobs_completed"
FROM "user"
JOIN "driver_on_shift" ON "user"."id" = "driver_on_shift"."driver_id"
JOIN "job" ON "driver_on_shift"."id" = "job"."driver_on_shift_id"
WHERE "job"."fact_end_at" >= CURRENT_DATE - INTERVAL '4 months'
AND "job"."status" = 'JOB_STATUS_COMPLETED'
GROUP BY "user"."id"
ORDER BY COUNT("job"."id") DESC
LIMIT 1;
"""

In [39]:
query = """
SELECT "user"."first_name", "user"."last_name", COUNT("job"."id") AS "total_jobs_completed"
FROM "user"
JOIN "driver_on_shift" ON "user"."id" = "driver_on_shift"."driver_id"
JOIN "job" ON "driver_on_shift"."id" = "job"."driver_on_shift_id"
WHERE "job"."fact_end_at" >= CURRENT_DATE - INTERVAL '4 months'
AND "job"."status" = 'JOB_STATUS_COMPLETED'
GROUP BY "user"."id"
ORDER BY COUNT("job"."id") DESC
LIMIT 1;
"""

In [ ]:
query = """
SELECT "user"."first_name", "user"."last_name", SUM("driver_on_shift"."odometer_reading") AS "total_distance"
FROM "user"
JOIN "driver_on_shift" ON "user"."id" = "driver_on_shift"."driver_id"
WHERE "driver_on_shift"."completed_at" >= CURRENT_DATE - INTERVAL '4 months'
GROUP BY "user"."first_name", "user"."last_name"
ORDER BY "total_distance" DESC
LIMIT 1;
"""

In [60]:
query = """
SELECT
    "job"."fact_start_at",
    COUNT("job"."id") AS "total_jobs_completed"
FROM
    "job"
JOIN
    "driver_on_shift" ON "job"."driver_on_shift_id" = "driver_on_shift"."id"
JOIN
    "user" ON "driver_on_shift"."driver_id" = "user"."id"
WHERE
    "user"."first_name" = 'Samat'
    AND "job"."fact_start_at" >= CURRENT_DATE - INTERVAL '3 months'
GROUP BY
    "job"."fact_start_at"
ORDER BY
    "job"."fact_start_at" DESC;
"""

In [45]:
conn=get_connection()

In [44]:
conn.close()

In [23]:
schema =get_schema_with_samples(conn)

In [24]:
print(schema)

Table: organization
Columns:
  - id (integer)
  - created_at (timestamp with time zone)
  - name (character varying)
  - type (character varying)
  - status (character varying)
Sample rows:
[('Resurs', 'ORGANIZATION_TYPE_SERVICE_PROVIDER', 'ORGANIZATION_STATUS_ACTIVE', 1, datetime.datetime(2025, 1, 31, 11, 58, 14, 296975, tzinfo=datetime.timezone.utc)), ('TOO Aspan Development', 'ORGANIZATION_TYPE_SERVICE_RECEIVER', 'ORGANIZATION_STATUS_ACTIVE', 2, datetime.datetime(2025, 1, 31, 11, 58, 15, 817477, tzinfo=datetime.timezone.utc)), ('Астанинский Ф.К. "B&A Contractors SA"', 'ORGANIZATION_TYPE_SERVICE_RECEIVER', 'ORGANIZATION_STATUS_ACTIVE', 3, datetime.datetime(2025, 1, 31, 11, 58, 15, 830681, tzinfo=datetime.timezone.utc))]

Table: apscheduler_jobs
Columns:
  - next_run_time (double precision)
  - job_state (bytea)
  - id (character varying)
Sample rows:
[('OffsetEtaUpdater', 1744191120.853555, <memory at 0x000001F422D050C0>)]

Table: job
Columns:
  - created_at (timestamp with time zone

In [ ]:
run_sql(query,conn)

{'columns': ['fact_start_at', 'total_jobs_completed'],
 'rows': [(datetime.datetime(2025, 4, 3, 9, 26, 17, 424395, tzinfo=datetime.timezone.utc),
   1),
  (datetime.datetime(2025, 4, 1, 7, 43, 58, 657698, tzinfo=datetime.timezone.utc),
   1),
  (datetime.datetime(2025, 4, 1, 6, 43, 52, 511750, tzinfo=datetime.timezone.utc),
   1),
  (datetime.datetime(2025, 3, 27, 12, 25, 22, 674040, tzinfo=datetime.timezone.utc),
   1),
  (datetime.datetime(2025, 3, 27, 12, 24, 10, 341528, tzinfo=datetime.timezone.utc),
   1),
  (datetime.datetime(2025, 3, 27, 12, 22, 57, 50099, tzinfo=datetime.timezone.utc),
   1),
  (datetime.datetime(2025, 3, 27, 11, 52, 6, 210150, tzinfo=datetime.timezone.utc),
   1),
  (datetime.datetime(2025, 3, 27, 10, 51, 54, 270465, tzinfo=datetime.timezone.utc),
   1),
  (datetime.datetime(2025, 3, 27, 6, 11, 1, 590746, tzinfo=datetime.timezone.utc),
   1),
  (datetime.datetime(2025, 3, 27, 5, 55, 6, 677560, tzinfo=datetime.timezone.utc),
   1),
  (datetime.datetime(2025, 3,

: 